In [ ]:
import os
import pandas as pd

input_folder = r"C:\Users\shenbagam\Downloads\FINALPROJECT\DataSet"
segment_folder = r"C:\Users\shenbagam\Downloads\FINALPROJECT\Segment"
os.makedirs(segment_folder, exist_ok=True)

input_file = os.path.join(input_folder, "fact_customers.csv")

column_groups = {
    "Customer_Identity": ["customer_id"],

    "Value_Revenue": [
        "last_invoice_amount_usd", "lifetime_revenue_usd",
        "add_on_revenue_usd", "next_quarter_revenue_usd",
        "acv_local", "acv_usd", "subscription_value_local"
    ],

    "Engagement_Usage": [
        "daily_active_users", "feature_adoption_pct",
        "sessions_per_user_day", "avg_session_minutes",
        "integrations_count", "custom_reports_count",
        "automations_active", "api_calls_monthly", "mobile_users_pct"
    ],

    "Satisfaction_Risk": [
        "support_tickets", "csat_score", "nps_score",
        "renewal_probability", "price_sensitivity", "renewal_risk_flag", "churn"
    ]
}

df = pd.read_csv(input_file, encoding="utf-8", low_memory=False, on_bad_lines="skip")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")


for group_name, cols in column_groups.items():
    group_folder = os.path.join(segment_folder, group_name)
    os.makedirs(group_folder, exist_ok=True)

    selected_cols = [col for col in cols if col in df.columns]
    if selected_cols:
        subset = df[selected_cols]
        output_file = os.path.join(group_folder, f"{group_name}.csv")
        subset.to_csv(output_file, index=False, encoding="utf-8-sig")
        print(f"✅ Saved {group_name} columns to {output_file} with shape {subset.shape}")
    else:
        print(f"⚠️ No matching columns found for {group_name}, skipping...")

all_selected = []
for cols in column_groups.values():
    all_selected.extend(cols)

selected_cols = [col for col in all_selected if col in df.columns]
combined_df = df[selected_cols]

if not combined_df.empty:
    combined_file = os.path.join(segment_folder, "all_segmentation_features.csv")
    combined_df.to_csv(combined_file, index=False, encoding="utf-8-sig")
    print(f"✅ Combined segmentation file saved at: {combined_file} with shape {combined_df.shape}")


✅ Saved Customer_Identity columns to C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\Customer_Identity\Customer_Identity.csv with shape (5150, 1)
✅ Saved Value_Revenue columns to C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\Value_Revenue\Value_Revenue.csv with shape (5150, 7)
✅ Saved Engagement_Usage columns to C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\Engagement_Usage\Engagement_Usage.csv with shape (5150, 9)
✅ Saved Satisfaction_Risk columns to C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\Satisfaction_Risk\Satisfaction_Risk.csv with shape (5150, 7)
✅ Combined segmentation file saved at: C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\all_segmentation_features.csv with shape (5150, 24)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

input_file = r"C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\all_segmentation_features.csv"
churn_file = r"C:\Users\shenbagam\Downloads\FINALPROJECT\Outputs\Phase4_Churn_Predictions.csv"
output_file = r"C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\cleaned_features.csv"

df = pd.read_csv(input_file, encoding="utf-8", low_memory=False, on_bad_lines="skip")
churn_df = pd.read_csv(churn_file)

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
churn_df.columns = churn_df.columns.str.strip().str.lower().str.replace(" ", "_")

if "customer_id" not in df.columns and "customerid" in df.columns:
    df.rename(columns={"customerid": "customer_id"}, inplace=True)
if "customer_id" not in churn_df.columns and "customerid" in churn_df.columns:
    churn_df.rename(columns={"customerid": "customer_id"}, inplace=True)

df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").fillna(0).astype(int)
churn_df["customer_id"] = pd.to_numeric(churn_df["customer_id"], errors="coerce").fillna(0).astype(int)

df = df[df["customer_id"].isin(churn_df["customer_id"])]
df = df.drop_duplicates(subset=["customer_id"])

threshold = 0.4
df = df.loc[:, df.isnull().mean() < threshold]

num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.median()))
df[cat_cols] = df[cat_cols].apply(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown"))

ordinal_mappings = {
    "account_tier": {"Bronze": 0, "Silver": 1, "Gold": 2, "Platinum": 3},
    "price_sensitivity": {"Low": 0, "Medium": 1, "High": 2},
    "health_score": {"Low": 0, "Medium": 1, "High": 2}
}

for col, mapping in ordinal_mappings.items():
    if col in df.columns:
        df[col] = df[col].map(mapping).fillna(0)

for col in cat_cols:
    if col not in ordinal_mappings:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

df.rename(columns={"customer_id": "Customer_Id"}, inplace=True)
df["Customer_Id"] = df["Customer_Id"].astype(int)

df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"✅ Cleaned dataset saved at: {output_file} with shape {df.shape}")
print("Customer_Id column type:", df["Customer_Id"].dtype)
print("Sample Customer_Id values:", df["Customer_Id"].head(10).tolist())


✅ Cleaned dataset saved at: C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\cleaned_features.csv with shape (4754, 24)
Customer_Id column type: int64
Sample Customer_Id values: [11177, 14771, 11091, 14818, 14376, 10280, 10997, 10585, 11486, 13927]


In [ ]:
import pandas as pd
import numpy as np

input_file = r"C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\all_segmentation_features_cleaned.csv"
output_file = r"C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\engineered_features.csv"

df = pd.read_csv(input_file)

numeric_cols = [
    "daily_active_users", "monthly_active_users", "sessions_per_user_day",
    "feature_adoption_pct", "avg_session_minutes",
    "lifetime_revenue_usd", "add_on_revenue_usd",
    "support_tickets", "renewal_probability", "renewal_risk_flag",
    "price_sensitivity", "csat_score", "nps_score"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

if {"daily_active_users", "monthly_active_users"}.issubset(df.columns):
    df["engagement_ratio"] = df["daily_active_users"] / df["monthly_active_users"].replace(0, np.nan)

if {"sessions_per_user_day", "monthly_active_users"}.issubset(df.columns):
    df["sessions_per_month"] = (df["sessions_per_user_day"] * 30) / df["monthly_active_users"].replace(0, np.nan)

weights = {
    "engagement_ratio": 0.4,
    "sessions_per_month": 0.3,
    "feature_adoption_pct": 0.3
}
df["engagement_score"] = sum(
    df[col] * w for col, w in weights.items() if col in df.columns
)

if {"lifetime_revenue_usd", "monthly_active_users"}.issubset(df.columns):
    df["revenue_per_user"] = df["lifetime_revenue_usd"] / df["monthly_active_users"].replace(0, np.nan)

if {"add_on_revenue_usd", "lifetime_revenue_usd"}.issubset(df.columns):
    df["addon_revenue_ratio"] = df["add_on_revenue_usd"] / df["lifetime_revenue_usd"].replace(0, np.nan)

if {"revenue_per_user", "addon_revenue_ratio"}.issubset(df.columns):
    df["revenue_level"] = df["revenue_per_user"].fillna(0) + df["addon_revenue_ratio"].fillna(0)


if {"support_tickets", "monthly_active_users"}.issubset(df.columns):
    df["ticket_rate"] = df["support_tickets"] / df["monthly_active_users"].replace(0, np.nan)

if {"renewal_probability", "renewal_risk_flag"}.issubset(df.columns):
    df["renewal_risk_score"] = (1 - df["renewal_probability"]) + df["renewal_risk_flag"]

risk_components = []
for col in ["renewal_risk_score", "price_sensitivity"]:
    if col in df.columns:
        risk_components.append(df[col].fillna(0))
if risk_components:
    df["churn_risk"] = sum(risk_components)


support_components = []
for col in ["ticket_rate", "csat_score", "nps_score"]:
    if col in df.columns:
        support_components.append(df[col].fillna(0))
if support_components:
    df["support_dependency"] = sum(support_components)

if "customer_id" in df.columns:
    df.rename(columns={"customer_id": "Customer_Id"}, inplace=True)

df["Customer_Id"] = pd.to_numeric(df["Customer_Id"], errors="coerce").fillna(0).astype(int)

df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"✅ Feature engineered dataset saved at: {output_file} with shape {df.shape}")
print("Customer_Id column type:", df["Customer_Id"].dtype)
print("Sample Customer_Id values:", df["Customer_Id"].head(10).tolist())


✅ Feature engineered dataset saved at: C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\engineered_features.csv with shape (5017, 29)
Customer_Id column type: int64
Sample Customer_Id values: [11177, 14771, 11091, 14818, 14376, 10280, 10997, 10585, 11486, 13927]


In [ ]:
import pandas as pd

input_file = r"C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\engineered_features.csv"
output_file = r"C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\encoded_features.csv"

df = pd.read_csv(input_file)

if "customer_id" in df.columns:
    df.rename(columns={"customer_id": "Customer_Id"}, inplace=True)

df["Customer_Id"] = pd.to_numeric(df["Customer_Id"], errors="coerce").fillna(0).astype(int)

categorical_nominal = [
    "contract_type", "legal_entity_type", "hq_vs_billing",
    "csm_assigned", "onboarding_completed", "qbr_completed_last_6m", "churn"
]

categorical_ordinal = {
    "account_tier": {"Bronze": 0, "Silver": 1, "Gold": 2, "Platinum": 3},
    "price_sensitivity": {"Low": 0, "Medium": 1, "High": 2},
    "health_score": {"Low": 0, "Medium": 1, "High": 2}
}


for col, mapping in categorical_ordinal.items():
    if col in df.columns:
        df[col] = df[col].map(mapping).fillna(0)

df_encoded = pd.get_dummies(
    df,
    columns=[col for col in categorical_nominal if col in df.columns],
    drop_first=True
)

df_encoded.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"✅ Encoded dataset saved at: {output_file} with shape {df_encoded.shape}")
print("Customer_Id column type:", df_encoded["Customer_Id"].dtype)
print("Sample Customer_Id values:", df_encoded["Customer_Id"].head(10).tolist())


✅ Encoded dataset saved at: C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\encoded_features.csv with shape (5017, 29)
Customer_Id column type: int64
Sample Customer_Id values: [11177, 14771, 11091, 14818, 14376, 10280, 10997, 10585, 11486, 13927]


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

input_file = r"C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\encoded_features.csv"
output_file = r"C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\scaled_features.csv"

df = pd.read_csv(input_file)

if "customer_id" in df.columns:
    df.rename(columns={"customer_id": "Customer_Id"}, inplace=True)

df["Customer_Id"] = pd.to_numeric(df["Customer_Id"], errors="coerce").fillna(0).astype(int)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

exclude_cols = ["Customer_Id", "geo_id", "industry_id", "product_id"]
numeric_cols = [col for col in numeric_cols if col not in exclude_cols]

print("Numeric columns to scale:", numeric_cols)

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df[numeric_cols])

df_scaled.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"✅ Scaled dataset saved at: {output_file} with shape {df_scaled.shape}")
print("Customer_Id column type:", df_scaled["Customer_Id"].dtype)
print("Sample Customer_Id values:", df_scaled["Customer_Id"].head(10).tolist())


Numeric columns to scale: ['last_invoice_amount_usd', 'lifetime_revenue_usd', 'add_on_revenue_usd', 'next_quarter_revenue_usd', 'acv_local', 'acv_usd', 'subscription_value_local', 'daily_active_users', 'feature_adoption_pct', 'sessions_per_user_day', 'avg_session_minutes', 'integrations_count', 'custom_reports_count', 'automations_active', 'api_calls_monthly', 'mobile_users_pct', 'support_tickets', 'csat_score', 'nps_score', 'renewal_probability', 'price_sensitivity', 'renewal_risk_flag', 'engagement_score', 'addon_revenue_ratio', 'renewal_risk_score', 'churn_risk', 'support_dependency']
✅ Scaled dataset saved at: C:\Users\shenbagam\Downloads\FINALPROJECT\Segment\scaled_features.csv with shape (5017, 29)
Customer_Id column type: int64
Sample Customer_Id values: [11177, 14771, 11091, 14818, 14376, 10280, 10997, 10585, 11486, 13927]
